In [ ]:
!pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 52.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import matplotlib.subplots as plt
import matplotlib.pyplot as plt
import pandas as pd
import gc
import optuna
from scipy.stats import skew, kurtosis
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
from sktime.transformations.panel.rocket import MiniRocketMultivariate

# Desabilitar logs do optuna para não poluir a saída
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. CARREGAMENTO E FILTRAGEM DOS CANAIS
# ==========================================
print("1. Carregando dados (Treino, Validação, Teste - já padronizados)...")
data_path = "/kaggle/input/datasets/lscolombo/hwu-usp/hwu_usp_windowed_data_v2.npz"
dados = np.load(data_path)

print("\n--- CARREGAMENTO DE TODOS OS SENSORES ---")
print("Utilizando todos os canais disponíveis no dataset.")
print("--------------------------------------------------\n")

# Formato original (samples, time_steps, channels) convertido para (samples, channels, time_steps)
X_train = np.transpose(dados['X_train'], (0, 2, 1)).astype(np.float32)
X_val   = np.transpose(dados['X_val'], (0, 2, 1)).astype(np.float32)
X_test  = np.transpose(dados['X_test'], (0, 2, 1)).astype(np.float32)

y_train = dados['y_train']
y_val   = dados['y_val']
y_test  = dados['y_test']

del dados
gc.collect()

# ==========================================
# 2. PREPARAÇÃO DOS DADOS E FEATURE ENGINEERING
# ==========================================
print("2. Preparando Features Multimodais...")

# Definição dos índices dos canais (Ajuste se necessário)
IDX_WEARABLE = slice(0, 13)  # Canais inerciais contínuos
IDX_AMBIENTE = slice(13, 25) # Canais discretos/binários

# --- A. FEATURES ESTATÍSTICAS MANUAIS (Separadas por modalidade) ---
print("Extraindo features estatísticas específicas por modalidade...")
def extract_multimodal_stats(X):
    # 1. Separar Modalidades
    X_wear = X[:, IDX_WEARABLE, :]
    X_amb  = X[:, IDX_AMBIENTE, :]

    # 2. Features Contínuas (Wearables)
    mean_w = np.mean(X_wear, axis=2)
    std_w  = np.std(X_wear, axis=2)
    skew_w = skew(X_wear, axis=2, nan_policy='omit')
    kurt_w = kurtosis(X_wear, axis=2, nan_policy='omit')

    features_wearable = np.hstack([mean_w, std_w, skew_w, kurt_w])

    # 3. Features Discretas (Ambiente)
    # Presença de evento (Houve algum 1 na janela?)
    max_a = np.max(X_amb, axis=2)

    # Tempo ativo (Soma de todos os 1s na janela)
    sum_a = np.sum(X_amb, axis=2)

    # Transições de estado (Quantas vezes mudou de 0 para 1 ou 1 para 0)
    trans_a = np.sum(np.abs(np.diff(X_amb, axis=2)), axis=2)

    features_ambiente = np.hstack([max_a, sum_a, trans_a])

    # 4. Unir tudo
    return np.hstack([features_wearable, features_ambiente])

X_train_stats = extract_multimodal_stats(X_train)
X_val_stats   = extract_multimodal_stats(X_val)
X_test_stats  = extract_multimodal_stats(X_test)

print("Padronizando as features manuais...")
scaler_stats = StandardScaler()
X_train_stats_scaled = scaler_stats.fit_transform(X_train_stats)
X_val_stats_scaled   = scaler_stats.transform(X_val_stats)
X_test_stats_scaled  = scaler_stats.transform(X_test_stats)

# --- B. MINIROCKET (APENAS NOS WEARABLES) ---
print("Extraindo features com MiniRocket (apenas canais contínuos)...")
minirocket = MiniRocketMultivariate()

# Damos o fit APENAS no recorte dos wearables
minirocket.fit(X_train[:, IDX_WEARABLE, :])

def transform_in_batches(model, X, batch_size=4000):
    n_samples = X.shape[0]
    features = []
    for i in range(0, n_samples, batch_size):
        fim = min(i + batch_size, n_samples)
        # Transforma APENAS o recorte dos wearables
        features.append(model.transform(X[i:fim, IDX_WEARABLE, :]).astype(np.float32))
    return np.vstack(features)

X_train_mr = transform_in_batches(minirocket, X_train)
X_val_mr   = transform_in_batches(minirocket, X_val)
X_test_mr  = transform_in_batches(minirocket, X_test)

# Libera RAM
del X_train, X_val, X_test
gc.collect()

# --- C. COMPOSIÇÃO DOS CENÁRIOS ---
# Sem FE Manual: Só o MiniRocket avaliando o movimento do braço
X_train_no_fe = X_train_mr
X_val_no_fe   = X_val_mr
X_test_no_fe  = X_test_mr

# Com FE Manual: Convoluções do Braço + Estatísticas do Braço + Eventos do Ambiente
X_train_fe = np.hstack([X_train_mr, X_train_stats_scaled])
X_val_fe   = np.hstack([X_val_mr, X_val_stats_scaled])
X_test_fe  = np.hstack([X_test_mr, X_test_stats_scaled])


# ==========================================
# 3. FUNÇÕES AUXILIARES (OPTUNA E GRID)
# ==========================================

def treinar_sem_optuna(X_t, y_t, X_v, y_v):
    alphas = np.logspace(-3, 3, 10)
    melhor_f1 = 0
    melhor_alpha = None
    for a in alphas:
        # Adicionando StandardScaler via pipeline
        clf = make_pipeline(StandardScaler(), RidgeClassifier(alpha=a))
        clf.fit(X_t, y_t)
        preds = clf.predict(X_v)
        f1 = f1_score(y_v, preds, average='macro')
        if f1 > melhor_f1:
            melhor_f1, melhor_alpha = f1, a
    return {'alpha': melhor_alpha, 'solver': 'auto'}

def treinar_com_optuna(X_t, y_t, X_v, y_v, n_trials=20):
    def objective(trial):
        a = trial.suggest_float('alpha', 1e-3, 1e3, log=True)
        # Reduzindo o espaço de busca do solver
        solver = trial.suggest_categorical('solver', ["auto", "svd", "cholesky", "lsqr", "sparse_cg"])

        # Adicionando StandardScaler via pipeline
        clf = make_pipeline(StandardScaler(), RidgeClassifier(alpha=a, solver=solver))
        clf.fit(X_t, y_t)
        preds = clf.predict(X_v)
        return f1_score(y_v, preds, average='macro')

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

# ==========================================
# 4. AVALIAÇÃO DOS 4 CENÁRIOS
# ==========================================
resultados = []

cenarios = [
    ("1. Sem FE Manual (Só MiniRocket) + Sem Optuna", X_train_no_fe, X_val_no_fe, X_test_no_fe, False),
    ("2. Sem FE Manual (Só MiniRocket) + Com Optuna", X_train_no_fe, X_val_no_fe, X_test_no_fe, True),
    ("3. Com FE Manual (MiniRocket + Stats) + Sem Optuna", X_train_fe, X_val_fe, X_test_fe, False),
    ("4. Com FE Manual (MiniRocket + Stats) + Com Optuna", X_train_fe, X_val_fe, X_test_fe, True)
]

print("\n3. Executando Cenários e Gerando Relatórios...")
for nome, Xt, Xv, Xte, usa_optuna in cenarios:
    print(f"\n======================================================")
    print(f"--- {nome} ---")

    if usa_optuna:
        best_params = treinar_com_optuna(Xt, y_train, Xv, y_val)
    else:
        best_params = treinar_sem_optuna(Xt, y_train, Xv, y_val)

    best_alpha = best_params['alpha']
    best_solver = best_params.get('solver', 'auto')

    print(f"Melhores parâmetros (Validação): Alpha={best_alpha:.4f}, Solver={best_solver}")

    # Treino final e teste com Pipeline
    clf_final = make_pipeline(StandardScaler(), RidgeClassifier(alpha=best_alpha, solver=best_solver))
    clf_final.fit(Xt, y_train)
    y_pred = clf_final.predict(Xte)

    acc_test = accuracy_score(y_test, y_pred)
    f1_test = f1_score(y_test, y_pred, average='macro')

    print(f"\nRelatório de Classificação ({nome}):")
    print(classification_report(y_test, y_pred))

    # Matriz de Confusão Normalizada
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues', normalize='true', ax=ax)
    plt.title(f"Matriz de Confusão Normalizada\n{nome}")
    plt.show()

    resultados.append({
        "Cenário": nome,
        "Acurácia": acc_test,
        "F1-Score (Macro)": f1_test
    })

# ==========================================
# 5. RESUMO FINAL E GRÁFICOS COMPARATIVOS
# ==========================================
print("\n=== TABELA COMPARATIVA DE DESEMPENHO ===")
df_resultados = pd.DataFrame(resultados)
display(df_resultados)

# Histograma / Gráfico de Barras comparando Acc e F1
df_resultados.set_index("Cenário").plot(kind='bar', figsize=(10, 6), colormap='viridis', width=0.7)
plt.title("Comparação de Acurácia e F1-Score entre Cenários")
plt.ylabel("Score (0 a 1)")
plt.xlabel("Cenários")
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1.1)
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
